# RF-DETR — Chinee apple weed detection (Colab)

Self-contained Colab notebook for the `rf-detr` experiment in `weed-detection-experiments`.

What it does:
1. Installs [`rfdetr`](https://github.com/roboflow/rf-detr) and `supervision`.
2. Loads `RFDETRBase` with its default COCO-pretrained weights.
3. Runs detection on an image and draws boxes + class labels.
4. Launches a Gradio UI with an adjustable confidence threshold.

**Caveat:** the default checkpoint detects COCO classes (80) — *not* Chinee apple. On weed imagery it will usually fire on `potted plant` or nothing above 0.5. This experiment is the scaffolding to swap a fine-tuned checkpoint into later; for now it shows what an off-the-shelf detector sees in the same image you ran through DINOv3.

**Before running:** Runtime → Change runtime type → **GPU** (T4 free is fine).

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Install dependencies

First run takes 1–2 minutes (rfdetr pulls transformers, timm, etc.). Model weights download from Hugging Face on first `RFDETRBase()` call.

In [ ]:
%pip install -q rfdetr supervision 'gradio>=4.0'

## 2. Configuration

Set `WEIGHTS` to a fine-tuned checkpoint path (e.g. on Drive) once you have one; leave as `None` to use the COCO baseline.

In [ ]:
WEIGHTS = None  # e.g. '/content/drive/MyDrive/rfdetr/weights/chinee_apple.pth'
DEFAULT_THRESHOLD = 0.3

## 3. Load model

In [ ]:
from rfdetr import RFDETRBase

model = RFDETRBase(pretrain_weights=WEIGHTS) if WEIGHTS else RFDETRBase()
print('Model ready.')

## 4. Inference code

In [ ]:
import supervision as sv
from PIL import Image
from rfdetr.util.coco_classes import COCO_CLASSES

_box = sv.BoxAnnotator()
_label = sv.LabelAnnotator()

def infer(image: Image.Image, threshold: float = DEFAULT_THRESHOLD) -> Image.Image:
    detections = model.predict(image, threshold=threshold)
    labels = [
        f"{COCO_CLASSES[cid]} {conf:.2f}"
        for cid, conf in zip(detections.class_id, detections.confidence)
    ]
    out = image.copy()
    out = _box.annotate(out, detections)
    out = _label.annotate(out, detections, labels)
    return out

## 5. Debug — list all detections

Upload the same image you used for DINOv3. Prints every detection (including low confidence) so you can see what RF-DETR considers there.

In [ ]:
import io
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

raw = model.predict(img, threshold=0.05)
print(f'Detections at threshold 0.05: {len(raw.xyxy)}')
for cid, conf, box in sorted(zip(raw.class_id, raw.confidence, raw.xyxy), key=lambda t: -t[1]):
    x0, y0, x1, y1 = [round(v) for v in box]
    print(f'  {COCO_CLASSES[cid]:>15s}  conf={conf:.3f}  box=({x0},{y0})-({x1},{y1})')

display(infer(img, threshold=DEFAULT_THRESHOLD))

## 6. Launch the Gradio UI

Click the public `*.gradio.live` URL. Use the slider to sweep the confidence threshold and see how detections change.

In [ ]:
import gradio as gr

gr.Interface(
    fn=infer,
    inputs=[
        gr.Image(type='pil', label='Upload'),
        gr.Slider(0.05, 0.95, value=DEFAULT_THRESHOLD, step=0.05, label='Confidence threshold'),
    ],
    outputs=gr.Image(type='pil', label='Detections'),
    title='RF-DETR object detection — Chinee apple (COCO baseline)',
    description='Default checkpoint is COCO-pretrained. Swap WEIGHTS in cell 2 to a fine-tuned Chinee apple checkpoint when available.',
).launch(share=True)